### LIMPIEZA DE PIMA-DIABETES

In [28]:
import pandas as pd
import numpy as np

# 1. Cargar el CSV crudo (sin encabezados)
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
df = pd.read_csv('../data/pima-indians-diabetes.csv', names=names)

print("Shape inicial:", df.shape)
print("Duplicados:", df.duplicated().sum())

# 2. Arreglar el bug del punto decimal en 'pedi'
# Cualquier valor entero es un valor con el punto decimal perdido (3 decimales originales)
mask_corrupto = df['pedi'] % 1 == 0
print(f"\nValores corruptos en 'pedi' detectados: {mask_corrupto.sum()}")
df.loc[mask_corrupto, 'pedi'] = df.loc[mask_corrupto, 'pedi'] / 1000

print("Rango de 'pedi' después de la corrección:", df['pedi'].min(), "-", df['pedi'].max())

# 3. Reemplazar ceros biológicamente imposibles por NaN
cols_con_ceros_invalidos = ['plas', 'pres', 'skin', 'test', 'mass']
for col in cols_con_ceros_invalidos:
    n_ceros = (df[col] == 0).sum()
    print(f"{col}: {n_ceros} ceros -> NaN")
    df[col] = df[col].replace(0, np.nan)

# 4. Imputar los NaN (mediana por columna, agrupada por clase para mayor precisión)
for col in cols_con_ceros_invalidos:
    df[col] = df.groupby('class')[col].transform(lambda x: x.fillna(x.median()))

# 5. Verificación final
print("\n" + "=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Shape final: {df.shape}")
print(f"\nMissing restantes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print("\nDataset limpio")

# 6. Guardar
df.to_csv("pima-indians-diabetes-clean.csv", index=False, header=False)

Shape inicial: (768, 9)
Duplicados: 0

Valores corruptos en 'pedi' detectados: 693
Rango de 'pedi' después de la corrección: 0.078 - 2.42
plas: 5 ceros -> NaN
pres: 35 ceros -> NaN
skin: 227 ceros -> NaN
test: 374 ceros -> NaN
mass: 11 ceros -> NaN

RESUMEN FINAL
Shape final: (768, 9)

Missing restantes:
Series([], dtype: int64)

Dataset limpio


<h2><font color="#004D7F" size=6>Módulo 3. Preprocesamiento de datos</font></h2>



<h1><font color="#004D7F" size=5>3. Métodos de remuestreo</font></h1>



---

<h2><font color="#004D7F" size=5>Índice</font></h2>
<a id="indice"></a>

* [1. Introducción](#section1)
    * [1.1. Librerías y CSV](#section11)
* [2. Validación cruzada](#section2)
    * [2.1. _k_-fold Cross Validation](#section21)
    * [2.2. Validación cruzada repetida](#section22)
    * [2.3. Validación cruzada dejando uno fuera](#section23)
* [3. División en porcentaje](#section3)
    * [3.1. División en porcentaje train/test](#section31)
    * [3.2. División train/test repetidos aleatoriamente](#section32)
* [4. Qué técnica usar](#section4)

In [16]:
# Permite ajustar la anchura de la parte útil de la libreta (reduce los márgenes)
# from IPython.core.display import display, HTML
from IPython.display import display, HTML
display(HTML("<style>.container{ width:98% }</style>"))

---
<a id="section1"></a>
# <font color="#004D7F"> 1. Introducción</font>

La evaluación es una estimación que podemos usar para hablar sobre qué tan bien creemos que el algoritmo realmente puede funcionar en la práctica. No es una garantía de rendimiento. Una vez que estimamos el rendimiento de nuestro algoritmo, podemos volver a entrenar el algoritmo final en todo el conjunto de datos de entrenamiento y prepararlo para su uso operativo. A continuación, veremos cuatro técnicas diferentes que podemos usar para dividir nuestro conjunto de datos de entrenamiento y crear estimaciones útiles de rendimiento para nuestros algoritmos de Machine Learning:
* Cómo dividir un conjunto de datos en subconjuntos por porcentaje para entrenamiento/validación.
* Cómo evaluar la robustez del modelo utilizando la validación cruzada, k-fold, con y sin repeticiones.
* Cómo evaluar la robustez del modelo usando una validación cruzada dejando uno fuera (LOOCV).
* División en train/test repetidos aleatoriamente.

<a id="section11"></a>
## <font color="#004D7F"> 1.1. Librerías y CSV</font>

Como siempre cargamos el CSV que vamos a utilizar. Así mismo, vamos a cargar las librerías principales que utilizaremos en esta sección.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# filename = "../data/pima-indians-diabetes.csv"
filename = "../data/pima-indians-diabetes-clean.csv"
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = pd.read_csv(filename, names=names)
array = data.values

X = array[:,0:8]
Y = array[:,8]

print(data.shape)

(768, 9)


<div style="text-align: right"> <font size=5>
    <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a>
</font></div>

---

<a id="section2"></a>
# <font color="#004D7F"> 2. Validación cruzada</font>

La validación cruzada es un proceso en la que se realizan $K$ particiones o **folds** de la base de datos y con ellos se realizan $K$ evaluaciones diferentes, de tal forma que todos los casos por lo menos una vez se encuentran en el conjunto de test. Básicamente en la evaluación $i$, la partición $i$ son los casos de test y el resto son los casos de entrenamiento. Finalmente, se realiza una media de los resultados obtenidos en las diferentes evaluaciones. En la siguiente imagen se ve un ejemplo de esto.

<img src="https://static.oschina.net/uploads/img/201609/26155106_OfXx.png" alt="cross-validation" width="500">

<a id="section21"></a>
## <font color="#004D7F"> 2.1. _k_-fold Cross Validation</font>

El método de validación cruzada de _k-fold_ implica dividir el conjunto de datos en _k_-particiones, también llamados _fold_. Cada subconjunto se mantiene mientras el modelo se entrena en todos los demás particiones. Este proceso se repite hasta que se determina la precisión para cada instancia en el conjunto de datos, y se proporciona una estimación de precisión general. Es un método robusto para estimar la precisión, y el tamaño de _k_ puede ajustar la cantidad de sesgo en la estimación, con valores populares establecidos en 5 y 10.

Puede ver que informamos tanto la media como la desviación estándar de la medida de rendimiento. Al resumir las medidas de rendimiento, es una buena práctica resumir la distribución de las medidas, en este caso suponiendo una distribución gaussiana del rendimiento (un supuesto muy razonable) y registrando la desviación estándar y media.

In [ ]:
# Evaluar con k-fold cross validation
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

num_folds = 10
seed = 7

# crea 10 particiones aleatorias (pero reproducibles gracias a random_state).
kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)

model = LogisticRegression(solver='lbfgs', max_iter=1000)

results = cross_val_score(model, X, Y, cv=kfold)

print(f"Accuracy: {results.mean() * 100.0:.2f}% ({results.std() * 100.0:.2f}%)")

# en resumen es la validacion cruzada k-fold:se divide el dataset en k partes,se entrena con k-1 y se evalua con la restante, repitiendo k veces y promediando resultados

Accuracy: 77.61% (3.75%)


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Puede obtener más información en la documentación oficial de Scikit-Learn sobre la clase [`KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html?highlight=kfold#sklearn.model_selection.KFold).
</div>

<a id="section22"></a>
## <font color="#004D7F"> 2.2. Validación cruzada repetida</font>

Una extensión de esta técnica de entrenamiento/validación es el poder repetir varias veces el proceso de dividir los datos _k-fold_. En este caso, la precisión final del modelo se toma como la media del número de repeticiones.

Al igual que el caso anterior podemos observar que nos informa tanto la media como la desviación estándar de la medida de rendimiento.

In [ ]:
# Evaluar usando Leave One Out Cross Validationfrom 
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

seed = 7
num_folds = 10
num_repeated = 5
#  hace 10-fold CV pero repetido 5 veces (50 evaluaciones en total).
repeatedcv = RepeatedKFold(
    n_splits=num_folds,
    n_repeats=num_repeated,
    random_state=seed
)

model = LogisticRegression(solver='lbfgs', max_iter=1000)

# calcula el accuracy en cada evaluacion y se promedia
results = cross_val_score(model, X, Y, cv=repeatedcv)

print(f"Accuracy: {results.mean() * 100.0:.2f}% ({results.std() * 100.0:.2f}%)")

# en resumen lo mismo que k-fold pero se repite varias veces con particiones diferentes, para tener una mejor precision

Accuracy: 77.27% (4.09%)


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Puede obtener más información en la documentación oficial de Scikit-Learn sobre la clase [`RepeatedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RepeatedKFold.html?highlight=kfold#sklearn.model_selection.RepeatedKFold).
</div>

<a id="section23"></a>
## <font color="#004D7F"> 2.3. Validación cruzada dejando uno fuera</font>

En Validación cruzada dejando uno fuera (Leave One Out Cross Validation, LOOCV), se omite una instancia de datos y se construye un modelo en todas las demás instancias de datos en el conjunto de entrenamiento, repitiéndose este proceso para todas las instancias de datos. El resultado es una gran cantidad de medidas de rendimiento que se pueden resumir en un esfuerzo por proporcionar una estimación más razonable del _Accuracy_ de su modelo.

Puede ver en la desviación estándar que la puntuación tiene más varianza que los resultados de validación cruzada _k-fold_ descritos anteriormente.

In [ ]:
# evaluar usando Leave One 
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# genera tantas particiones como filas en el dataset, dejando una fila para evaluar y entrenando con el resto
loocv = LeaveOneOut()
model = LogisticRegression(solver='lbfgs', max_iter=1000)
# evalua cada una (predice 1 dato por iteracion)
results = cross_val_score(model, X, Y, cv=loocv)
print(f"Accuracy:{results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

# # # > preciso que el anterior, pero mucho mas costoso computacionalmente, ya que se entrena k veces (n = numero de instancias), cada vez se deja una instancia para evaluar y se entrena con el resto, luego se promedia el accuracy de todas las instancias < sesgo & > varianza 

Accuracy:77.21% (41.95%)


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Puede obtener más información en la documentación oficial de Scikit-Learn sobre la clase [`LeaveOneOut`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html?highlight=leaveoneout#sklearn.model_selection.LeaveOneOut).
</div>

<div style="text-align: right"> <font size=5>
    <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a>
</font></div>

---

<a id="section3"></a>
# <font color="#004D7F"> 3. División en porcentaje</font>

<a id="section31"></a>
## <font color="#004D7F"> 3.1. División en porcentaje train/test</font>

La división de datos implica la partición de los datos en un conjunto de datos de entrenamiento explícito utilizado para preparar el modelo y un conjunto de datos de validación invisible (se dice validación invisible ya que aunque se conozca el atributo clase el modelo lo omitirá para una vez realizada la predicción compararlo para ver si ha acertado) que se utiliza para evaluar el rendimiento del modelo en datos no etiquetados.

En el ejemplo a continuación, dividimos el conjunto de datos de los indios Pima en divisiones de 67%/33% para entrenamiento y prueba y evaluar el Accuracy de un modelo Logistic Regression.

In [ ]:
# evaluar usando a train and a test set
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size, random_state=seed)
model = LogisticRegression(solver='lbfgs', max_iter=1000)
model.fit(X_train, Y_train)
result = model.score(X_test, Y_test)
print(f"Accuracy:{result*100.0:,.2f}%")

# en pocas palabras, se divide el dataset en dos partes(entrenar && evaluar) 

Accuracy:78.35%


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Puede obtener más información en la documentación oficial de Scikit-Learn sobre la clase [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html?highlight=train_test_split#sklearn.model_selection.train_test_split).
</div>

<a id="section32"></a>
## <font color="#004D7F"> 3.2. División train/test repetidos aleatoriamente</font>

Otra variación en la validación cruzada de k-fold es crear una división aleatoria de los datos como la división de train/test descrita anteriormente, pero repetir el proceso de división y evaluación del algoritmo varias veces, como la validación cruzada. El siguiente ejemplo divide los datos en una división de train/test del 67%/33% y repite el proceso 10 veces.

Podemos ver que en este caso la distribución de la medida de desempeño está a la par con _k_-fold cross validation pero reducimos la varianza considerablemente.

In [ ]:
# evaluar usando Shuffle Split Cross Validation
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

n_splits = 10
test_size = 0.33
seed = 7
kfold = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
model = LogisticRegression(solver='lbfgs', max_iter=1000)
results = cross_val_score(model, X, Y, cv=kfold)
print(f"Accuracy:{results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

# similar a train/test split, pero se repite varias veces con particiones diferentes, para tener una mejor precision

Accuracy:76.77% (2.11%)


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Puede obtener más información en la documentación oficial de Scikit-Learn sobre la clase [`ShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html?highlight=shufflesplit#sklearn.model_selection.ShuffleSplit).
</div>

<div style="text-align: right"> <font size=5>
    <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a>
</font></div>

---

<a id="section4"></a>
# <font color="#004D7F"> 4. Qué técnica usar</font>

Esta sección enumera algunos consejos para considerar qué técnica de remuestreo usar en diferentes circunstancias.
* Por lo general, la validación cruzada de _k-fold_ es el estándar defacto para evaluar el rendimiento de un algoritmo en datos no etiquetados con _k_ configurado en 3, 5 o 10.
* El uso de una división de train/test es buena para la velocidad cuando se usa un algoritmo lento y produce estimaciones de rendimiento con un sesgo más bajo cuando se usan conjuntos de datos grandes.
* Las técnicas como LOOCV y las divisiones aleatorias repetidas pueden ser intermedios útiles cuando se trata de equilibrar la variación en el rendimiento estimado, la velocidad de entrenamiento del modelo y el tamaño del conjunto de datos.

El mejor consejo es experimentar y encontrar una técnica para su problema que sea rápida y produzca estimaciones razonables de rendimiento que pueda usar para tomar decisiones. En caso de duda, utilice la validación cruzada con un _k-fold_ de 10.

<div style="text-align: right"> <font size=5>
    <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a>
</font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>

## USANDO DATASET MARMARA2

In [ ]:
# lIMPIEAZA DE MARMARA2
import pandas as pd

df = pd.read_csv('../data/marmara2.csv')

print("Shape inicial:", df.shape)

# Verificacion general (por si acaso)
print("\nNulos por columna:\n", df.isnull().sum())
print("\nDuplicados encontrados:", df.duplicated().sum())

# Eliminar la fila duplicada
df = df.drop_duplicates()

print("\n" + "=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Shape final: {df.shape}")
print(f"\nMissing restantes:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print("\nDataset limpio")

# Guardar
df.to_csv("marmara_clean.csv", index=False)

Shape inicial: (1760, 7)

Nulos por columna:
 total_length_cm    0
body_depth_cm      0
body_width_cm      0
head_length_cm     0
eye_diameter_mm    0
weight_gr          0
species            0
dtype: int64

Duplicados encontrados: 1

RESUMEN FINAL
Shape final: (1759, 7)

Missing restantes:
Series([], dtype: int64)

Dataset limpio


In [ ]:
import pandas as pd
from numpy import mean, std
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (KFold, RepeatedKFold, LeaveOneOut,
                                      cross_val_score, train_test_split, ShuffleSplit)


# Cargar el dataset ya limpio
df = pd.read_csv('../data/marmara_clean.csv')

# Separar predictoras (X) y variable objetivo (Y)
feature_cols = ['total_length_cm', 'body_depth_cm', 'body_width_cm',
                 'head_length_cm', 'eye_diameter_mm', 'weight_gr']
X = df[feature_cols].values

# Codificar 'species' (texto) a numeros, ya que los modelos necesitan valores numéricos
le = LabelEncoder()
Y = le.fit_transform(df['species'])

print("Clases:", dict(zip(le.classes_, range(len(le.classes_)))))
df.head()

Clases: {'Hamsi': 0, 'Kalkan': 1, 'Lüfer': 2, 'Mezgit': 3, 'Palamut': 4, 'Sardalya': 5, 'Tekir': 6, 'Zargana': 7, 'İstavrit': 8, 'İzmarit': 9}


,total_length_cm,body_depth_cm,body_width_cm,head_length_cm,eye_diameter_mm,weight_gr,species
0,7.9,1.2,0.5,1.9,2.2,3.2,Hamsi
1,13.7,2.1,1.0,2.9,3.4,15.0,Hamsi
2,18.5,3.8,1.9,4.4,4.9,47.9,Sardalya
3,12.1,2.3,1.1,3.0,3.7,15.7,İstavrit
4,7.6,1.4,0.6,1.8,1.9,3.5,İstavrit


In [ ]:
# se usa pipeline porque sklearn recalcula el escalado usando solo los datos de entrenamiento, y luego aplica ese escalado a los datos de prueba, evitando el data leakage(fuga de datos- el modelo ve datos que no deberia ver)
# usando k-fold cross validation con escalamiento de variables y regresion logistica
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ShuffleSplit, cross_val_score

n_splits = 10
test_size = 0.33
seed = 7

kfold = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)

# Pipeline: primero se escala, luego entrena el modelo
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver='lbfgs', max_iter=1000)
)

results = cross_val_score(model, X, Y, cv=kfold)
print(f"Accuracy: {results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

# se escala para que todas las variables tengan la misma importancia, ya que algunas tienen rangos muy diferentes (ej. eye_diameter_mm vs total_length_cm), pero se escala después de separar los datos de entrenamiento y prueba

Accuracy: 73.22% (0.92%)


In [ ]:
# usando k-fold cross repetida  
from sklearn.model_selection import RepeatedKFold

kfold = RepeatedKFold(n_splits=10, n_repeats=3, random_state=7)
model = make_pipeline(StandardScaler(), LogisticRegression(solver='lbfgs', max_iter=1000))
results = cross_val_score(model, X, Y, cv=kfold)
print(f"Accuracy: {results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

Accuracy: 74.32% (3.20%)


In [ ]:
# usando validacion cruzada uno fuera - Leave One Out (LOOCV)
from sklearn.model_selection import LeaveOneOut

loocv = LeaveOneOut()
model = make_pipeline(StandardScaler(), LogisticRegression(solver='lbfgs', max_iter=1000))
results = cross_val_score(model, X, Y, cv=loocv)
print(f"Accuracy: {results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

Accuracy: 74.76% (43.44%)


In [ ]:
# usando train and a test set
from sklearn.model_selection import train_test_split

test_size = 0.33
seed = 7
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=test_size, random_state=seed, stratify=Y)

model = make_pipeline(StandardScaler(), LogisticRegression(solver='lbfgs', max_iter=1000))
model.fit(X_train, Y_train)
result = model.score(X_test, Y_test)
print(f"Accuracy: {result*100.0:,.2f}%")

Accuracy: 71.26%


In [ ]:
# usando train and test pero con Shuffle Split Cross Validation
from sklearn.model_selection import ShuffleSplit

n_splits = 10
test_size = 0.33
seed = 7
kfold = ShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
model = make_pipeline(StandardScaler(), LogisticRegression(solver='lbfgs', max_iter=1000))
results = cross_val_score(model, X, Y, cv=kfold)
print(f"Accuracy: {results.mean()*100.0:,.2f}% ({results.std()*100.0:,.2f}%)")

Accuracy: 73.22% (0.92%)
